# 第 16 课｜自相关与预测

> **数据身份**：固定随机种子生成的模拟教学数据，不是真实证券行情。  
> **环境**：`pkm-quant` · Python 3.11 · 默认离线运行。  
> **课程原始材料**：`16_youtube-xKjtlaYwWr0/index.html`  
> **所属模块**：P1-M3 · 统计证据与可预测性

## 本节量化问题

历史收益与未来收益的线性依赖是否稳定、可交易？

## 准确定义与策略报告位置

ACF 描述不同滞后下的线性相关；Ljung–Box 检验一组滞后相关是否整体为零。

- **核心关系**：ρ_k=Corr(R_t,R_(t-k))；Ljung–Box 联合检验前若干阶自相关是否为零。
- **写入报告**：滞后结构、预测信号和白噪声检验
- **P2 衔接**：P2 第 36、49–50 课：rolling、统计模型和时序切分。


## 运行环境与数据契约

先确认项目、数据目录和数据身份。后续联网数据源必须通过适配层进入，不能改变本 Notebook 的研究计算口径。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## ACF 与 Ljung–Box 联合诊断

ACF 单独看每个滞后，Ljung–Box 检查多个滞后是否整体为零。统计显著仍需成本后验证。

In [2]:
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import acf

daily = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])
market = daily.loc[daily["symbol"] == "SIM_MARKET"].set_index("date")["adj_close"]
returns = market.pct_change(fill_method=None).dropna()

acf_values = acf(returns, nlags=10, fft=True)
ljung_box = acorr_ljungbox(returns, lags=[5, 10], return_df=True)

print("ACF lag 0–10:", np.round(acf_values, 4))
display(ljung_box)

ACF lag 0–10: [ 1.      0.0468 -0.0206 -0.0135 -0.0137 -0.0201 -0.0035  0.0565 -0.0025
  0.0208  0.0209]


,lb_stat,lb_pvalue
5,2.573776,0.765346
10,5.693668,0.840310


## 方法边界与阶段验收

显著自相关可能来自微观结构、重叠收益或数据处理；还需样本外和成本验证。

完成本课后应能：

1. 不看课程材料准确解释核心概念和公式符号；
2. 用小样本手工核对 Notebook 的关键结果；
3. 说明结果能支持什么判断、不能支持什么判断；
4. 指出它在策略报告中的位置，以及仍缺少的数据、成本或稳健性证据。
